# AIAP Mini Project 1

**Dataset:** `mini_project_1_data.csv`  

---

In [ ]:
# Setup Terminal Formatting Constants & Standard Libraries
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# User defined ANSI color constants for terminal print output
RED_BOLD = "\033[1;31m"
GREEN_BOLD = "\033[1;32m"
CLEAR = "\033[0m"

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print(f"{GREEN_BOLD}[INFO] Environment initialized and formatting constants defined.{CLEAR}")

## 1. Preliminary Data Exploration

Examining raw structure, non-null counts, datatypes, missingness, duplicates, and summary statistics.

In [ ]:
# Load raw dataset with binary offset handling
with open('./data/mini_project_1_data.csv', 'rb') as f:
    content = f.read()

header_start = content.find(b'ID,URL')
data_bytes = content[header_start:]
text = data_bytes.decode('latin1')

lines = text.splitlines()
csv_lines = [line for line in lines if line.startswith(('ID,', '0,', '1,', '2,', '3,', '4,', '5,', '6,', '7,', '8,', '9,')) or (len(line) > 0 and line[0].isdigit())]

csv_text = '\n'.join(csv_lines)
df = pd.read_csv(io.StringIO(csv_text))
df.columns = [c.strip().replace('\\', '') for c in df.columns]

print(f"{GREEN_BOLD}[INFO] Raw Dataset Shape: {df.shape}{CLEAR}")
df.head()

In [ ]:
# Non-null inspection, missing value finding, and duplicate detection
print(f"{GREEN_BOLD}[INFO] Dataset Structure Overview:{CLEAR}")
df.info()

missing_counts = df.isnull().sum()[df.isnull().sum() > 0]
print(f"\n{RED_BOLD}[FINDING] Features with Missing Values:\n{missing_counts}{CLEAR}")
print(f"\n{RED_BOLD}[WARNING] Total Duplicate Rows Detected: {df.duplicated().sum()}{CLEAR}")

In [ ]:
# Descriptive Statistics
df.describe().T

## 2. Univariate Analysis

Analyzing categorical frequencies and continuous feature skews.

In [ ]:
# Categorical Distributions: weekday & data_channel
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fix warning 1: explicit hue assignment + legend=False
sns.countplot(data=df, x='weekday', hue='weekday', order=['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'], ax=axes[0], palette='viridis', legend=False)
axes[0].set_title('Article Count by Weekday')

sns.countplot(data=df, x='data_channel', hue='data_channel', ax=axes[1], palette='magma', legend=False)
axes[1].set_title('Article Count by Data Channel')

plt.tight_layout()
plt.show()

In [ ]:
# Target Variable Distribution: 'shares' Raw vs Log Transformed
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df['shares'], bins=50, kde=True, ax=axes[0], color='salmon')
axes[0].set_title('Raw Shares Distribution (Skewed)')

sns.histplot(np.log1p(df['shares']), bins=50, kde=True, ax=axes[1], color='teal')
axes[1].set_title('Log(Shares + 1) Transformed')

plt.tight_layout()
plt.show()

## 3. Bivariate Analysis & Correlation

Examining relations between predictors and `shares` using Pearson linear and Spearman rank correlation coefficients.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
pearson_r = df[num_cols].corr(method='pearson')['shares']
spearman_rho = df[num_cols].corr(method='spearman')['shares']

corr_df = pd.DataFrame({'Pearson_r': pearson_r, 'Spearman_rho': spearman_rho}).sort_values('Spearman_rho', ascending=False)
print(f"{GREEN_BOLD}[INFO] Spearman Rank vs Pearson Linear Correlation Matrix:{CLEAR}")
corr_df

## 4. Data Cleaning Pipeline Execution

Executing systematic deduplication, imputation, and column dropping.

In [ ]:
# Execute Cleaning Operations
cleaned_df = df.copy()

# 1. Deduplication
initial_rows = len(cleaned_df)
cleaned_df.drop_duplicates(inplace=True)
dedup_count = initial_rows - len(cleaned_df)
print(f"{RED_BOLD}[FINDING] Removed {dedup_count} duplicated rows.{CLEAR}")

# 2. Missing Value Imputation (Fix warning 2: avoiding chained assignment with .fillna)
cleaned_df['num_videos'] = cleaned_df['num_videos'].fillna(0)
cleaned_df['data_channel'] = cleaned_df['data_channel'].fillna('Unknown')
cleaned_df['num_imgs'] = cleaned_df['num_imgs'].fillna(cleaned_df['num_imgs'].median())

for col in ['num_hrefs', 'num_self_hrefs', 'self_reference_min_shares', 'self_reference_max_shares', 'self_reference_avg_shares']:
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

# 3. Drop Identifiers
cleaned_df.drop(columns=['ID', 'URL'], inplace=True)

print(f"{GREEN_BOLD}[INFO] Cleaned Dataset Shape: {cleaned_df.shape}{CLEAR}")
print(f"{GREEN_BOLD}[INFO] Total Remaining Missing Values: {cleaned_df.isnull().sum().sum()}{CLEAR}")

## 5. Feature Engineering

Constructing and transforming domain-relevant features based on news article characteristics before model training [cite: 10].

In [ ]:
# 1. Create Feature Engineering transformations on cleaned_df
fe_df = cleaned_df.copy()

# Ratio of self links to total links (avoiding division by zero)
fe_df['self_link_ratio'] = fe_df['num_self_hrefs'] / (fe_df['num_hrefs'] + 1e-5)

# Article media density (images + videos)
fe_df['total_media'] = fe_df['num_imgs'] + fe_df['num_videos']

# Log transform target 'shares' to stabilize variance during modeling
fe_df['log_shares'] = np.log1p(fe_df['shares'])

print(f"{GREEN_BOLD}[INFO] Feature Engineering Completed. Total columns: {fe_df.shape[1]}{CLEAR}")
fe_df[['self_link_ratio', 'total_media', 'log_shares']].head()

## 6. Data Splitting

Separating features ($X$) and target ($y$), followed by Train/Validation/Test splitting to prevent data leakage [cite: 10].

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Separate Features (X) and Target (y)
# We drop both raw 'shares' and log-transformed 'log_shares' from X
X = fe_df.drop(columns=['shares', 'log_shares'])
y = fe_df['log_shares']

# 2. Split into Train (80%) and Temporary Holdout (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Split Temporary Holdout into Validation (10%) and Test (10%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"{GREEN_BOLD}[INFO] Training Set Shape:   X_train = {X_train.shape}, y_train = {y_train.shape}{CLEAR}")
print(f"{GREEN_BOLD}[INFO] Validation Set Shape: X_val   = {X_val.shape}, y_val   = {y_val.shape}{CLEAR}")
print(f"{GREEN_BOLD}[INFO] Test Set Shape:       X_test  = {X_test.shape}, y_test  = {y_test.shape}{CLEAR}")

## 7. Feature Scaling & Encoding Pipeline

Constructing `ColumnTransformer` preprocessing pipelines for numerical scaling and categorical encoding [cite: 10].

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify categorical and numerical columns dynamically
nominal_features = ['weekday', 'data_channel']
numerical_features = [col for col in X.columns if col not in nominal_features]

# Create transformer pipelines
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('nom', nominal_transformer, nominal_features)
    ],
    remainder='passthrough',
    n_jobs=-1
)

print(f"{GREEN_BOLD}[INFO] ColumnTransformer pipeline initialized successfully.{CLEAR}")
preprocessor

## 8. Model Development

Implementing baseline linear regression models, regularized regression models (Ridge & Lasso), and model evaluation metrics following Section 1.4 of the AIAP Study Guide.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Helper function to calculate evaluation metrics
def evaluate_regression_model(model, X_data, y_true, dataset_name, model_name):
    y_pred = model.predict(X_data)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    metrics = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}
    print(f"{GREEN_BOLD}[INFO] {model_name} ({dataset_name} Evaluation):{CLEAR}")
    for k, v in metrics.items():
        print(f"   {k:5s}: {v:.6f}")
    return metrics

print(f"{GREEN_BOLD}[INFO] Evaluation helper initialized.{CLEAR}")

### 8.1 Baseline Multivariate Linear Regression

Chaining preprocessor and unregularized `LinearRegression` into an `sklearn` Pipeline.

In [ ]:
# Build Multivariate Linear Regression Pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Fit on Training Data
lr_pipeline.fit(X_train, y_train)

# Evaluate on Validation Set
lr_val_metrics = evaluate_regression_model(lr_pipeline, X_val, y_val, 'Validation', 'Multivariate Linear Regression')

### 8.2 Ridge Regression (L2 Regularization)

Applying L2 penalty shrinkage (`alpha=1.0`) to handle feature multicollinearity.

In [ ]:
# Build Ridge Regression Pipeline
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0, random_state=42))
])

# Fit on Training Data
ridge_pipeline.fit(X_train, y_train)

# Evaluate on Validation Set
ridge_val_metrics = evaluate_regression_model(ridge_pipeline, X_val, y_val, 'Validation', 'Ridge Regression')

### 8.3 Lasso Regression (L1 Regularization)

Applying L1 penalty (`alpha=0.01`) for built-in feature sparsity and selection.

In [ ]:
# Build Lasso Regression Pipeline
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=0.01, random_state=42))
])

# Fit on Training Data
lasso_pipeline.fit(X_train, y_train)

# Evaluate on Validation Set
lasso_val_metrics = evaluate_regression_model(lasso_pipeline, X_val, y_val, 'Validation', 'Lasso Regression')

### 8.4 Baseline Model Performance Comparison

Comparing validation metrics across Multivariate Linear, Ridge, and Lasso Regression.

In [ ]:
# Summary comparison table
metrics_df = pd.DataFrame({
    'Multivariate Linear': lr_val_metrics,
    'Ridge (alpha=1.0)': ridge_val_metrics,
    'Lasso (alpha=0.01)': lasso_val_metrics
}).T

print(f"{GREEN_BOLD}[INFO] Baseline Model Validation Metrics Summary:{CLEAR}")
metrics_df

## 9. Hyperparameter Tuning

Fine-tuning regularization parameters (alpha) using `GridSearchCV` and `RandomizedSearchCV`, followed by final evaluation on the holdout test set following Section 1.5 of the AIAP Study Guide.

### 9.1 Grid Search CV

Systematically searching across a grid of alpha values using 5-fold cross-validation on the training set.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Define hyperparameter grid for Ridge and Lasso
param_grid = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

# 2. Grid Search CV for Ridge Regression
ridge_grid = GridSearchCV(ridge_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
ridge_grid.fit(X_train, y_train)

# 3. Grid Search CV for Lasso Regression
lasso_grid = GridSearchCV(lasso_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
lasso_grid.fit(X_train, y_train)

print(f"{GREEN_BOLD}[INFO] Best Ridge Hyperparameters (GridSearch): {ridge_grid.best_params_}{CLEAR}")
print(f"{GREEN_BOLD}[INFO] Best Ridge CV R2 Score: {ridge_grid.best_score_:.6f}{CLEAR}")
print(f"{GREEN_BOLD}[INFO] Best Lasso Hyperparameters (GridSearch): {lasso_grid.best_params_}{CLEAR}")
print(f"{GREEN_BOLD}[INFO] Best Lasso CV R2 Score: {lasso_grid.best_score_:.6f}{CLEAR}")

### 9.2 Randomized Search CV

Sampling random alpha parameters from a uniform distribution to optimize search efficiency.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# Define continuous distribution for alpha
param_dist = {
    'regressor__alpha': uniform(loc=0.001, scale=10.0)
}

# Randomized Search CV for Ridge
ridge_rand = RandomizedSearchCV(ridge_pipeline, param_distributions=param_dist, n_iter=10, cv=5, scoring='r2', random_state=42, n_jobs=-1)
ridge_rand.fit(X_train, y_train)

print(f"{GREEN_BOLD}[INFO] Best Ridge Hyperparameters (RandomSearch): {ridge_rand.best_params_}{CLEAR}")
print(f"{GREEN_BOLD}[INFO] Best Ridge CV R2 Score: {ridge_rand.best_score_:.6f}{CLEAR}")

### 9.3 Final Model Selection & Holdout Test Set Evaluation

Evaluating the single best tuned model on the unseen holdout test set (`X_test`, `y_test`) for unbiased performance reporting.

In [ ]:
# Select best estimator from Grid Search
best_model = ridge_grid.best_estimator_

# Evaluate on Final Unseen Test Set
test_metrics = evaluate_regression_model(best_model, X_test, y_test, 'Final Test Set', 'Tuned Ridge Model')

print(f"{GREEN_BOLD}[INFO] Final Model Evaluation complete!{CLEAR}")